In [1]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

import fitz  # PyMuPDF
import re
import unicodedata
from langchain_core.documents import Document
import numpy as np

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_38232\144695714.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyMuPDFLoader
f:\Udemy\youtube\rag\.venv-rag\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [2]:
DB_NAME = str(Path("vector_db_2_2"))
KNOWLEDGE_BASE = str(Path("knowledge-base_2_2"))

DB_NAME, KNOWLEDGE_BASE

('vector_db_2_2', 'knowledge-base_2_2')

In [3]:
embeddings = OllamaEmbeddings(model="embeddinggemma-2:570m")

In [4]:
# ตั้งเป็น False ถ้าไม่ต้องการ OCR

ENABLE_OCR_FALLBACK = True


_ZERO_WIDTH = re.compile(r"[\u200b\u200c\u200d\u2060\ufeff\x00]")

# ช่องว่างที่ PDF มักแทรกก่อนสระบน/ล่าง/วรรณยุกต์/ไม้ไต่คู้ ฯลฯ

_SPACE_BEFORE_MARK = re.compile(r"\s+([\u0e31\u0e34-\u0e3a\u0e47-\u0e4e])")

_MULTI_NEWLINE = re.compile(r"\n{3,}")


_ocr_reader = None



def clean_thai_text(text: str) -> str:

    text = unicodedata.normalize("NFC", text)

    text = text.replace("\u0e4d\u0e32", "\u0e33")  # ํ + า -> ำ

    text = _ZERO_WIDTH.sub("", text)

    text = _SPACE_BEFORE_MARK.sub(r"\1", text)

    text = re.sub(r"[ \t]+", " ", text)

    return _MULTI_NEWLINE.sub("\n\n", text).strip()



def looks_garbled(text: str) -> bool:

    """ว่างเปล่า / มี (cid:xx) / มี � มากผิดปกติ -> น่าจะอ่านไม่ได้"""

    stripped = text.strip()

    if len(stripped) < 20:

        return True

    if "(cid:" in stripped:

        return True

    return stripped.count("\ufffd") / len(stripped) > 0.02



def ocr_page(page, dpi: int = 200) -> str:

    global _ocr_reader

    if _ocr_reader is None:

        import easyocr

        _ocr_reader = easyocr.Reader(["th", "en"], gpu=False)  # มี GPU ให้ใช้ gpu=True


    pix = page.get_pixmap(dpi=dpi, colorspace=fitz.csRGB, alpha=False)

    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)

    lines = _ocr_reader.readtext(img, detail=0, paragraph=True)

    return "\n".join(lines)



def load_pdf(path: str, doc_type: str) -> list[Document]:

    docs = []

    try:

        pdf = fitz.open(path)

    except Exception as e:

        print(f"[skip] เปิด PDF ไม่ได้: {path} ({e})")

        return docs


    with pdf:

        total = len(pdf)

        for i, page in enumerate(pdf):

            text = page.get_text("text", sort=True)

            method = "text"


            if looks_garbled(text) and ENABLE_OCR_FALLBACK:

                try:

                    text = ocr_page(page)

                    method = "ocr"

                except Exception as e:

                    print(f"[warn] OCR ไม่สำเร็จ {path} หน้า {i + 1}: {e}")


            text = clean_thai_text(text)

            if not text:

                continue


            docs.append(

                Document(

                    page_content=text,

                    metadata={

                        "source": path,

                        "page": i,

                        "total_pages": total,

                        "extract_method": method,

                        "doc_type": doc_type,

                    },

                )

            )

    return docs

In [5]:
def fetch_documents():

    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))

    documents = []


    # utf-8-sig จัดการ BOM ของไฟล์ไทยที่เซฟจาก Windows/Notepad

    text_patterns = ["**/*.txt", "**/*.text", "**/*.md"]

    text_kwargs = {"encoding": "utf-8-sig", "autodetect_encoding": True}


    for folder in folders:

        if not os.path.isdir(folder):

            continue

        doc_type = os.path.basename(folder)


        # ---- text / markdown ----

        for pattern in text_patterns:

            loader = DirectoryLoader(

                folder,

                glob=pattern,

                loader_cls=TextLoader,

                loader_kwargs=text_kwargs,

                silent_errors=True,

            )

            for doc in loader.load():

                doc.page_content = clean_thai_text(doc.page_content)

                doc.metadata["doc_type"] = doc_type

                documents.append(doc)


        # ---- pdf (รองรับ .pdf / .PDF) ----

        for pdf_path in Path(folder).rglob("*"):

            if pdf_path.suffix.lower() == ".pdf":

                documents.extend(load_pdf(str(pdf_path), doc_type))


    return documents

In [6]:
documents = fetch_documents()
print(f"Number of documents: {len(documents)}")
print(f"-"*100)
print(f"First document: \n{documents[:5]}")

Using CPU. Note: This module is much faster with a GPU.
f:\Udemy\youtube\rag\.venv-rag\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(
f:\Udemy\youtube\rag\.venv-rag\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Number of documents: 30
----------------------------------------------------------------------------------------------------
First document: 
[Document(metadata={'source': 'knowledge-base_2_2\\books\\VI - ลงทุนหุ้นให้รวย สไตล์ วอเร็น บัฟเฟตต์_Part2.pdf', 'page': 0, 'total_pages': 10, 'extract_method': 'ocr', 'doc_type': 'books'}, page_content='5. คุณรับความเสี่ยงได้มากแค่ไหน? การตัดสินใจจัดสรรการลงทุน\n49\nเ วิธีสร้างพอร์ตการลงทุนที่ดี\n67\n 7 การดูแลรักษาพอร์ตการลงทุน\n83\n8. คุณควรจ้างที่ปรึกษาการเงินหรือไม่?\n97\nด การเอาชนะ เกมชีวิต"\n113\nบทสรุป\n121\nกฎการลงทุนอย่างรอบคอบ 30 ข้อ\n125\nเชิงอรรถ\n135\nแหล่งอ้างอิง\n137\nกิตติกรรมประกาศ\n138'), Document(metadata={'source': 'knowledge-base_2_2\\books\\VI - ลงทุนหุ้นให้รวย สไตล์ วอเร็น บัฟเฟตต์_Part2.pdf', 'page': 1, 'total_pages': 10, 'extract_method': 'ocr', 'doc_type': 'books'}, page_content='บทนา:\nทำไม...ผมจึงเขียนหนังสือเล่มนี้ ?\nนังสือทีเป็นผลงานของผมทัง 12 เล่ม แต่ละเล่มจะเป็นเรื่องที่ผม v มักจะเขียนถึงเรื่อง "วิทยาศาสตร์การล

In [7]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [9]:
chunks = create_chunks(documents)
print(f"Number of chunks: {len(chunks)}")
print(f"-"*100)
print(f"First chunk: \n{chunks[0].page_content}")
print(f"-"*100)
print(f"Second chunk overlap with first chunk: \n{chunks[2].page_content}")

Number of chunks: 103
----------------------------------------------------------------------------------------------------
First chunk: 
5. คุณรับความเสี่ยงได้มากแค่ไหน? การตัดสินใจจัดสรรการลงทุน
49
เ วิธีสร้างพอร์ตการลงทุนที่ดี
67
 7 การดูแลรักษาพอร์ตการลงทุน
83
8. คุณควรจ้างที่ปรึกษาการเงินหรือไม่?
97
ด การเอาชนะ เกมชีวิต"
113
บทสรุป
121
กฎการลงทุนอย่างรอบคอบ 30 ข้อ
125
เชิงอรรถ
135
แหล่งอ้างอิง
137
กิตติกรรมประกาศ
138
----------------------------------------------------------------------------------------------------
Second chunk overlap with first chunk: 
นังสือทีเป็นผลงานของผมทัง 12 เล่ม แต่ละเล่มจะเป็นเรื่องที่ผม v มักจะเขียนถึงเรื่อง "วิทยาศาสตร์การลงทุน" เซึ่งเป็นหลักฐานที่ได้ รับการพิสูจน์แล้วในด้านกลยุทธ์การลงทุนฺในหนัง เสือชุด the only guides you ll ever need จะพูดถึงการลงทุนในหุ้น พันธบัตร ทางเลือกในการ ลงทนแ เละการวางแผนการเงินที่ถูกต้อง ส่วนชุด wise investing จะเป็น เรื่องเล่าเชิงอุปมาอุปไมยที่เปรียบเทียบกรณีบางอย่างกับการลงทุนฺฺซึ่งแสดง ให้เห็นว่ากลยุทธ์การลงทุนที่สามารถ

In [10]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [11]:
create_embeddings(chunks)
print("Embeddings created successfully")

There are 103 vectors with 768 dimensions in the vector store
Embeddings created successfully
